In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path

if "notebooks" in os.getcwd():
    os.chdir(Path.cwd().parent)

In [ ]:
import logging

logger = logging.getLogger("ts_visualization")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter("%(levelname)s: %(message)s")
    handler.setFormatter(formatter)
    logger.addHandler(handler)

In [ ]:
import numpy as np
from lets_plot import LetsPlot

from xaitimesynth import (
    TimeSeriesBuilder,
    constant,
    gaussian,
    list_components,
    list_feature_components,
    list_signal_components,
    manual,
    peak,
    plot_component,
    plot_components,
    random_walk,
    trend,
    gaussian_pulse,
)
from xaitimesynth.generators import (
    generate_ecg_like,
    generate_random_walk,
)

# Enable lets_plot for notebook display
LetsPlot.setup_html()

## Cylinder-Bell-Funnel dataset

The **Cylinder-Bell-Funnel (CBF)** dataset (Saito, 1994) is a classic synthetic benchmark. All three classes share the same Gaussian noise background ε(t) ~ N(0,1), but differ by a class-specific pattern added to a random window [a, b]:

| Class | Pattern inside [a, b] |
|-------|----------------------|
| **Cylinder** (0) | Constant level shift of amplitude (6 + η) |
| **Bell** (1) | Linearly increasing ramp: 0 → (6 + η) |
| **Funnel** (2) | Linearly decreasing ramp: (6 + η) → 0 |

Here η ~ N(0,1) is a per-sample amplitude noise. The window length is drawn from Uniform[32, 96] timesteps, placed at a random start position.

`generate_cylinder_bell_funnel()` wraps the builder to produce a ready-made CBF dataset including ground-truth `feature_masks` that mark exactly where [a, b] lies — enabling direct XAI evaluation.

In [ ]:
# Per-sample generators — η is drawn from rng so amplitude varies per sample
def cylinder_generator(n_timesteps, rng, length, **kwargs):
    eta = rng.randn()
    return np.full(length, 6.0 + eta)  # constant plateau


def bell_generator(n_timesteps, rng, length, **kwargs):
    eta = rng.randn()
    return np.linspace(0, 6.0 + eta, length)  # ramp up


def funnel_generator(n_timesteps, rng, length, **kwargs):
    eta = rng.randn()
    return np.linspace(6.0 + eta, 0, length)  # ramp down


dataset_manual = (
    TimeSeriesBuilder(
        n_timesteps=128, n_samples=300, normalization="none", random_state=42
    )
    .for_class(0)  # Cylinder
    .add_signal(gaussian(mu=0, sigma=1))
    .add_feature(
        manual(generator=cylinder_generator),
        random_location=True,
        length_pct=(0.25, 0.75),  # window length ~ Uniform[32, 96] timesteps
    )
    .for_class(1)  # Bell
    .add_signal(gaussian(mu=0, sigma=1))
    .add_feature(
        manual(generator=bell_generator), random_location=True, length_pct=(0.25, 0.75)
    )
    .for_class(2)  # Funnel
    .add_signal(gaussian(mu=0, sigma=1))
    .add_feature(
        manual(generator=funnel_generator),
        random_location=True,
        length_pct=(0.25, 0.75),
    )
    .build()
)

plot_components(dataset_manual, x_order=-1)

### Under the hood: building CBF manually with `manual()`

`generate_cylinder_bell_funnel()` is a thin wrapper around `TimeSeriesBuilder`. The key ingredient is `manual(generator=...)`, which lets you supply a **per-sample generator function** that receives the shared `rng` — necessary here because the amplitude noise η must be drawn fresh for every sample.

The variable window length is expressed with `length_pct=(0.25, 0.75)`: the builder samples a length uniformly from that fraction range for each sample independently.

In [ ]:
from xaitimesynth import generate_cylinder_bell_funnel

dataset = generate_cylinder_bell_funnel(random_state=42)

plot_components(dataset, x_order=-1)

# Helper functions

## Quickstart

In [ ]:
# TODO: I need to doublecheck whether this actually represents the cylinder-bell-funnel dataset well
# it probably doesn't right now - I remember the framework needed some more features to work it out fully

# 2 class dataset with 2 different types of features, all with gaussian background noise
gauss_sigma = 0.5

# Intialise dataset builder with data parameters
dataset_builder = (
    TimeSeriesBuilder(
        n_timesteps=250,
        n_samples=100,
        random_state=123,
        normalization="none",
    )
    # class 0: positive gaussian peak
    .for_class(0)
    .add_signal(gaussian(sigma=gauss_sigma))
    .add_feature(gaussian_pulse(amplitude=2), length_pct=0.25, random_location=True)
    # class 1: negative gaussian peak (with background noise)
    .for_class(1)
    .add_signal(gaussian(sigma=gauss_sigma))
    .add_feature(gaussian_pulse(amplitude=-2), length_pct=0.25, random_location=True)
)

# Create the dataset
dataset = dataset_builder.build()

plot_components(dataset_builder.build()).show()

# Synth Data

In [ ]:
# see generators/components already defined in package
print(list_components())
print(list_signal_components())
print(list_feature_components())

In [ ]:
list_components().keys()

print(list_signal_components().keys())
print(list_feature_components().keys())

## CDPE table visualisations

In [ ]:
# from xaitimesynth.generators import (
#     generate_gaussian_pulse,
#     generate_trend,
#     generate_seasonal,
#     generate_constant,
# )
# from lets_plot import geom_hline, theme_void, ggsize, theme, element_blank, ggsave
# from typing import Optional, Tuple


# def make_simple_plot(
#     signal: np.ndarray,
#     title: str,
#     plot_size: Optional[Tuple[int, int]] = (150, 100),
# ) -> "LetsPlotPlot":
#     """Create a simple Lets-Plot visualization for a 1D signal.

#     Args:
#         signal (np.ndarray): The time series signal to plot.
#         title (str): Title for the plot.
#         plot_size (Optional[Tuple[int, int]]): (width, height) in pixels.

#     Returns:
#         LetsPlotPlot: The configured plot object.
#     """
#     p = (
#         plot_component(signal, line_size=2)
#         + geom_hline(yintercept=0, size=1, linetype="dashed", color="gray")
#         + theme_void()
#         + ggsize(*(plot_size or (150, 100)))
#         + theme(title=element_blank())
#     )
#     return p


# # Gaussian pulse
# timesteps = 100
# amplitude = 0.5
# signal_gaussian_pulse = generate_gaussian_pulse(
#     timesteps, amplitude=amplitude, width_ratio=1, center=0.5
# )
# plot_gaussian_pulse = make_simple_plot(signal_gaussian_pulse, "Gaussian Pulse")

# # Trend
# signal_trend = generate_trend(timesteps, endpoints=(0, amplitude))
# plot_trend = make_simple_plot(signal_trend, "Trend")

# # Seasonal
# signal_seasonal = generate_seasonal(timesteps, period=10, amplitude=amplitude)
# plot_seasonal = make_simple_plot(signal_seasonal, "Seasonal")

# # Constant
# signal_constant = generate_constant(timesteps, value=amplitude)
# plot_constant = make_simple_plot(signal_constant, "Constant")

# # Show plots
# plot_gaussian_pulse.show()
# plot_trend.show()
# plot_seasonal.show()
# plot_constant.show()

# ggsave(plot_gaussian_pulse, "gaussian_pulse.pdf")
# ggsave(plot_trend, "trend.pdf")
# ggsave(plot_seasonal, "seasonal.pdf")
# ggsave(plot_constant, "constant.pdf")

## Single components

In [ ]:
signal = generate_random_walk(n_timesteps=1000, step_size=0.1)
plot_component(signal)


rng = np.random.default_rng(42)
signal = generate_random_walk(1000, step_size=0.1, rng=rng)
plot_component(signal)

In [ ]:
plot_component(component_type="red_noise", n_timesteps=1000, phi=0.99)

In [ ]:
plot_component(generate_ecg_like(400)).show()
plot_component(component_type="ecg_like", n_timesteps=400, line_color="blue").show()

In [ ]:
# Generate and plot a random walk
plot_component(component_type="random_walk", n_timesteps=100, step_size=0.2).show()

# Generate and plot a seasonal pattern
plot_component(
    component_type="seasonal", n_timesteps=200, period=20, amplitude=1.5
).show()

# Example 1: Using a manual component with custom values

# Create custom values for a zigzag pattern
zigzag = np.concatenate([np.linspace(0, 1, 10), np.linspace(1, 0, 10)])
zigzag = np.tile(zigzag, 5)  # Repeat the pattern 5 times

# Plot using the manual component with custom values
plot_component(
    component_type="manual",
    values=zigzag,
    n_timesteps=100,
    line_color="red",
).show()


# Define a custom generator function for a damped oscillation
def damped_oscillation(length, rng, frequency=0.1, decay=0.05, **kwargs):
    t = np.arange(length)
    return np.exp(-decay * t) * np.sin(2 * np.pi * frequency * t)


# Plot using the manual component with custom generator
plot_component(
    component_type="manual",
    generator=damped_oscillation,
    frequency=0.05,  # Lower frequency
    decay=0.02,  # Slower decay
    n_timesteps=200,
    line_color="blue",
    normalization="minmax",  # Show actual values
).show()

plot_component(
    component_type="manual",
    generator=damped_oscillation,
    frequency=0.05,  # Lower frequency
    decay=0.02,  # Slower decay
    n_timesteps=200,
    line_color="blue",
    normalization="minmax",  # Show actual values
    normalization_kwargs={"feature_range": (-1, 1)},  # Custom normalization range
).show()

## Univariate time series

### Cylinder-Bell-Funnel-like dataset creation example

In [ ]:
# TODO: I need to doublecheck whether this actually represents the cylinder-bell-funnel dataset well
# it probably doesn't right now - I remember the framework needed some more features to work it out fully

# Cylinder-Bell-Funnel-like dataset
gauss_sigma = 1
dataset_builder = (
    TimeSeriesBuilder(
        n_timesteps=128, n_samples=200, random_state=42, normalization="none"
    )
    # cylinder
    .for_class(0)
    .add_signal(gaussian(sigma=gauss_sigma))
    .add_feature(constant(value=6), length_pct=0.25, random_location=True)
    # bell
    .for_class(1)
    .add_signal(gaussian(sigma=gauss_sigma))
    .add_feature(trend(endpoints=[0, 6]), length_pct=0.25, random_location=True)
    # funnel
    .for_class(2)
    .add_signal(gaussian(sigma=gauss_sigma))
    .add_feature(trend(endpoints=[6, 0]), length_pct=0.25, random_location=True)
)
dataset = dataset_builder.build()
dataset
plot_components(dataset_builder.build()).show()

In [ ]:
# Create a custom component and use it
def my_seasonal_pattern(n_timesteps, rng, frequency=0.1, amplitude=1.0, **kwargs):
    """Generate a custom seasonal pattern."""
    t = np.arange(n_timesteps)
    return amplitude * np.sin(2 * np.pi * frequency * t / n_timesteps)


custom_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=10)
    .for_class(0)
    .add_signal(manual(generator=my_seasonal_pattern, frequency=0.05, amplitude=1.0))
    .add_signal(gaussian(sigma=0.1))
    .for_class(1)
    .add_signal(manual(generator=my_seasonal_pattern, frequency=0.05, amplitude=1.0))
    .add_signal(gaussian(sigma=0.1))
    .add_feature(peak(amplitude=0.5, width=5), start_pct=0.4, end_pct=0.6)
    .build()
)

plot_components(custom_dataset).show()

## Multivariate time series

In [ ]:
# Create a 3-dimensional time series dataset
multivariate_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=50, n_dimensions=3, random_state=42)
    # Class 0: Random walk in all dimensions but no discriminative features
    .for_class(0)
    .add_signal(random_walk(step_size=0.2), dim=[0, 1, 2])  # Apply to all dimensions
    .add_signal(gaussian(sigma=0.1), dim=[0, 1, 2])
    # Class 1: Features in different dimensions
    .for_class(1)
    .add_signal(random_walk(step_size=0.2), dim=[0, 1, 2])
    .add_signal(gaussian(sigma=0.1), dim=[0, 1, 2])
    # Add shapelet only to dimensions 0 and 1 with shared location (same position in both)
    .add_feature(
        constant(value=1.2),
        random_location=True,
        length_pct=0.15,
        dim=[0, 1],
        shared_location=True,
    )
    # Add peak only to dimension 2
    .add_feature(
        peak(amplitude=1.0, width=1),
        random_location=True,
        length_pct=0.05,
        dim=[0, 1, 2],
    )
    # Add level change to all dimensions, with different locations for each
    # .add_feature(
    #     level_change(amplitude=0.7),
    #     random_location=True,
    #     length_pct=0.2,
    #     dim=[0, 1, 2],
    #     shared_location=False,
    # )
    .build()
)

print(f"Dataset shape: {multivariate_dataset['X'].shape}")  # Should be (50, 100, 3)
print(f"Number of dimensions: {multivariate_dataset['metadata']['n_dimensions']}")

# Example of accessing a specific dimension
dim0_data = multivariate_dataset["X"][:, :, 0]  # First dimension for all samples
print(f"Dimension 0 shape: {dim0_data.shape}")

# Convert to DataFrame with specific dimensions
df = TimeSeriesBuilder().to_df(multivariate_dataset, dimensions=[0, 1])
print(df.head())

plot_components(multivariate_dataset)

In [ ]:
# For a single plot or list with one element
viz = plot_components(multivariate_dataset, dimensions=[0])
viz.show()

# For multiple plots
vizs = plot_components(multivariate_dataset, dimensions=[0, 1])
for i, viz in enumerate(vizs):
    print(f"Dimension {i}:")
    viz.show()

## Creating Train/Test/Validation Splits with clone()

The clone method allows creating multiple datasets from the same builder with different parameters.

In [ ]:
# Create a base builder with class definitions
base_builder = (
    TimeSeriesBuilder(n_timesteps=100, random_state=42)
    .for_class(0)  # Class 0: Just random walk with noise
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1))
    # Class 1: Random walk with noise plus features
    .for_class(1)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1))
    .add_feature(constant(value=1.0), start_pct=0.4, end_pct=0.6)
)

train_dataset = base_builder.clone(n_samples=140, random_state=42).build()
test_dataset = base_builder.clone(n_samples=40, random_state=43).build()
val_dataset = base_builder.clone(n_samples=20, random_state=44).build()

print(
    f"Train dataset: X shape={train_dataset['X'].shape}, y shape={train_dataset['y'].shape}"
)
print(
    f"Test dataset: X shape={test_dataset['X'].shape}, y shape={test_dataset['y'].shape}"
)
print(
    f"Validation dataset: X shape={val_dataset['X'].shape}, y shape={val_dataset['y'].shape}"
)

# Compare with explainer indexes
# This is now much simpler since each dataset has its own indexing starting from 0
print("\nSample index 2 in training set is truly sample index 2 (not an offset)")
print("Sample index 2 in test set is truly sample index 2 (not an offset)")

## Metrics